# LaMa (Hugging Face opencv/inpainting_lama) -> TensorFlow Lite (Android Uyumlu) Dönüştürme Notebook'u

Bu notebook, **Hugging Face `opencv/inpainting_lama` ONNX** modelini Android'de kullanılabilir bir `.tflite` modele dönüştürür ve doğrulama yapar:

1. Model lisansını kontrol eder (ticari kullanım uygunluğu doğrulaması).
2. ONNX modeli Hugging Face deposundan indirir.
3. ONNX Runtime ile referans çıktı üretir.
4. `onnx2tf` ile SavedModel'e çevirir.
5. TensorFlow Lite'a çevirir (SELECT_TF_OPS etkin).
6. TFLite çıktısını ONNX çıktısıyla karşılaştırır.
7. Android entegrasyonu için gerekli kontrolleri ve Gradle bağımlılıklarını verir.


In [ ]:
# Hücre 1 - Gereken paketler
!pip -q install --upgrade pip
!pip -q install onnx onnxruntime onnxsim tensorflow==2.15.0 onnx2tf==1.17.10 numpy pillow


In [ ]:
# Hücre 2 - Çalışma klasörü, model repo ve lisans kontrolü
from pathlib import Path
import json
import urllib.request

WORKDIR = Path('lama_tflite_export')
WORKDIR.mkdir(exist_ok=True)

HF_REPO = 'opencv/inpainting_lama'
HF_API_URL = f'https://huggingface.co/api/models/{HF_REPO}'

# Depoda ONNX dosya adı farklı olabilir; sıralı olarak denenecek.
ONNX_CANDIDATE_URLS = [
    f'https://huggingface.co/{HF_REPO}/resolve/main/inpainting_lama.onnx',
    f'https://huggingface.co/{HF_REPO}/resolve/main/lama.onnx',
    f'https://huggingface.co/{HF_REPO}/resolve/main/lama_fp32.onnx',
    f'https://huggingface.co/{HF_REPO}/resolve/main/model.onnx',
]

ONNX_PATH = WORKDIR / 'inpainting_lama.onnx'
SAVED_MODEL_DIR = WORKDIR / 'saved_model'
TFLITE_PATH = WORKDIR / 'inpainting-lama-android.tflite'

COMMERCIAL_FRIENDLY_LICENSES = {
    'apache-2.0',
    'mit',
    'bsd-2-clause',
    'bsd-3-clause',
    'cc-by-4.0',
}

def get_hf_license(api_url: str) -> str:
    with urllib.request.urlopen(api_url) as resp:
        info = json.loads(resp.read().decode('utf-8'))
    return (info.get('cardData', {}).get('license') or info.get('license') or '').lower().strip()

license_id = get_hf_license(HF_API_URL)
print('Hugging Face license:', license_id if license_id else 'belirtilmemiş')

if not license_id:
    raise RuntimeError('Model kartında lisans bilgisi bulunamadı. Ticari kullanım öncesi manuel doğrulama gerekli.')

if license_id not in COMMERCIAL_FRIENDLY_LICENSES:
    raise RuntimeError(
        f'Lisans ({license_id}) ticari kullanım için otomatik olarak uygun listede değil. '         'Hukuki inceleme yapmadan devam etmeyin.'
    )

print('✅ Lisans kontrolü geçti (otomatik allowlist).')


In [ ]:
# Hücre 3 - Hugging Face ONNX modelini indir
import urllib.request
from urllib.error import HTTPError, URLError

if not ONNX_PATH.exists():
    print('Model indiriliyor...')
    last_error = None
    for url in ONNX_CANDIDATE_URLS:
        try:
            print('Deneniyor:', url)
            urllib.request.urlretrieve(url, ONNX_PATH)
            print('İndirildi:', ONNX_PATH)
            break
        except (HTTPError, URLError) as e:
            last_error = e
            print('Başarısız:', e)
    else:
        raise RuntimeError(f'ONNX dosyası indirilemedi. Son hata: {last_error}')
else:
    print('Model zaten mevcut:', ONNX_PATH)

print('ONNX boyutu (MB):', round(ONNX_PATH.stat().st_size / 1024 / 1024, 2))


In [ ]:
# Hücre 4 - ONNX giriş/çıkışlarını incele
import onnx

model = onnx.load(str(ONNX_PATH))
print('IR version:', model.ir_version)
print('\nInputs:')
for i in model.graph.input:
    shape = [d.dim_value if d.dim_value > 0 else 'dynamic' for d in i.type.tensor_type.shape.dim]
    print('-', i.name, shape, i.type.tensor_type.elem_type)

print('\nOutputs:')
for o in model.graph.output:
    shape = [d.dim_value if d.dim_value > 0 else 'dynamic' for d in o.type.tensor_type.shape.dim]
    print('-', o.name, shape, o.type.tensor_type.elem_type)


In [ ]:
# Hücre 5 - ONNX Runtime ile referans inference
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])

print('Input names:', [i.name for i in sess.get_inputs()])
print('Input shapes:', [i.shape for i in sess.get_inputs()])

H, W = 512, 512
image = np.random.rand(1, 3, H, W).astype(np.float32)
mask = (np.random.rand(1, 1, H, W) > 0.5).astype(np.float32)

feed = {}
for inp in sess.get_inputs():
    c = inp.shape[1] if isinstance(inp.shape[1], int) else None
    if c == 3:
        feed[inp.name] = image
    elif c == 1:
        feed[inp.name] = mask
    else:
        feed[inp.name] = image

onnx_out = sess.run(None, feed)[0]
print('ONNX output shape:', onnx_out.shape, 'dtype:', onnx_out.dtype)


In [ ]:
# Hücre 6 - ONNX -> TensorFlow SavedModel
import shutil
import subprocess

if SAVED_MODEL_DIR.exists():
    shutil.rmtree(SAVED_MODEL_DIR)

cmds = [
    ['onnx2tf', '-i', str(ONNX_PATH), '-o', str(SAVED_MODEL_DIR)],
    ['onnx2tf', '-i', str(ONNX_PATH), '-o', str(SAVED_MODEL_DIR), '--disable_group_convolution'],
]

for idx, cmd in enumerate(cmds, start=1):
    print(f'Konvert denemesi {idx}:', ' '.join(cmd))
    ret = subprocess.run(cmd, capture_output=True, text=True)
    print(ret.stdout[-2000:])
    if ret.returncode == 0:
        print('SavedModel dönüşümü başarılı.')
        break
    print('Hata:', ret.stderr[-2000:])
else:
    raise RuntimeError('onnx2tf dönüşümü başarısız oldu. stderr logunu kontrol edin.')


In [ ]:
# Hücre 7 - SavedModel -> TFLite (Android dostu ayarlar)
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_MODEL_DIR))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

print('TFLite yazıldı:', TFLITE_PATH)
print('TFLite boyutu (MB):', round(TFLITE_PATH.stat().st_size / 1024 / 1024, 2))


In [ ]:
# Hücre 8 - TFLite inference + ONNX ile karşılaştırma
import numpy as np
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()

in_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()

print('TFLite inputs:')
for d in in_details:
    print('-', d['name'], d['shape'], d['dtype'])

for d in in_details:
    c = d['shape'][-1]
    if c == 3:
        interpreter.set_tensor(d['index'], np.transpose(image, (0,2,3,1)).astype(np.float32))
    elif c == 1:
        interpreter.set_tensor(d['index'], np.transpose(mask, (0,2,3,1)).astype(np.float32))
    else:
        raise ValueError(f'Beklenmeyen giriş shape: {d["shape"]}')

interpreter.invoke()
tflite_out = interpreter.get_tensor(out_details[0]['index'])

onnx_nhwc = np.transpose(onnx_out, (0,2,3,1)) if (onnx_out.ndim == 4 and onnx_out.shape[1] in (1,3)) else onnx_out

mae = np.mean(np.abs(onnx_nhwc - tflite_out))
max_err = np.max(np.abs(onnx_nhwc - tflite_out))
print(f'MAE: {mae:.6f}')
print(f'MAX_ERR: {max_err:.6f}')

assert mae < 0.03, f'MAE çok yüksek: {mae}'
print('✅ Sayısal doğrulama geçti.')


In [ ]:
# Hücre 9 - Android uyumluluk kontrolü
import tensorflow as tf

analysis = tf.lite.experimental.Analyzer.analyze(model_path=str(TFLITE_PATH))
print(analysis)

print('\nAndroid Gradle bağımlılıkları:')
print('implementation "org.tensorflow:tensorflow-lite:2.15.0"')
print('implementation "org.tensorflow:tensorflow-lite-select-tf-ops:2.15.0"')


## Android kullanım notları

- Giriş tensörlerini **NHWC float32** formatında verin:
  - `image`: `[1, H, W, 3]` (0..1 normalize)
  - `mask`: `[1, H, W, 1]` (0 veya 1)
- Çıkış tensörü tipik olarak `[1, H, W, 3]` olur.
- `SELECT_TF_OPS` içerdiği için Android tarafında `tensorflow-lite-select-tf-ops` bağımlılığı zorunludur.
